In [ ]:
# Basic imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Optional: make plots look nicer
sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
from pathlib import Path

feat_path = Path("../data/raw/UNSW-NB15_features.csv")
data_path1 = Path("../data/raw/UNSW-NB15_1.csv")
data_path2 = Path("../data/raw/UNSW-NB15_2.csv")

# Read the features file
feat_df = pd.read_csv(feat_path, encoding="ISO-8859-1")
feat_df.head()

In [ ]:
col_names = feat_df["Name"].tolist()
df1 = pd.read_csv(data_path1, names=col_names)
df2 = pd.read_csv(data_path2, names=col_names)
df = pd.concat([df1, df2], ignore_index=True)

# Show first few rows
display(df.head())

# Check shape
print("Dataset shape:", df.shape)

# Info about data types and missing values
display(df.info())

# Quick statistics for numeric columns
display(df.describe())

In [ ]:
# For visualisation
df_eda = df # use the full dataset

# Quick peek
display(df_eda.head())

# Check shape
print("Sample shape:", df_eda.shape)

In [ ]:
# Plot distribution of normal vs attack traffic
sns.countplot(x="Label", data=df_eda)
plt.title("Normal vs Attack Traffic (Sample)")
plt.show()

In [ ]:
# Plot scaled distribution of "sbytes" by label
df_eda["log_sbytes"] = np.log1p(df_eda["sbytes"])
sns.boxplot(x="Label", y="log_sbytes", data=df_eda)
plt.title("Distribution of Log(Source Bytes) by Label")
plt.show()

In [ ]:
# Plot scaled distribution of "dbytes" by label
df_eda["log_dbytes"] = np.log1p(df_eda["dbytes"])
sns.boxplot(x="Label", y="log_dbytes", data=df_eda)
plt.title("Distribution of Log(Destination Bytes) by Label")
plt.show()

In [ ]:
# Scatter relationship between "sbytes" and "dbytes"
plt.figure(figsize=(8, 6))
sns.scatterplot(x="sbytes", y="dbytes", hue="Label", data=df_eda, alpha=0.6)
plt.title("sbytes vs dbytes by Label")
plt.show()

# Highlight unusual byte combinations
df_eda['log_byte_diff'] = abs(np.log1p(df_eda['sbytes'] - np.log1p(df_eda['dbytes'])))
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="log_byte_diff", data=df_eda)
plt.title("Distribution of Log(Byte Difference) by Label")
plt.show()

# Downweight bidirectional network flows
df_eda["bidirectional_strength"] = np.log1p(
    np.minimum(df_eda["sbytes"], df_eda["dbytes"])
)
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="bidirectional_strength", data=df_eda)
plt.title("Distribution of Bidirectional Strength by Label")
plt.show()

In [ ]:
# Plot scaled distribution of "dur" by label
df_eda["log_dur"] = np.log1p(df_eda["dur"])
sns.boxplot(x="Label", y="log_dur", data=df_eda)
plt.title("Log Duration by Normal vs Attack Traffic (Sample)")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_eda,
    x="log_sbytes",
    y="log_dbytes",
    hue="Label",
    alpha=0.6,
    palette={0: "blue", 1: "red"}
)
plt.xlabel("Log(Source Bytes)")
plt.ylabel("Log(Destination Bytes)")
plt.title("Source vs Destination Bytes (Sampled Flows)")
plt.show()

In [ ]:
df_eda["byte_ratio"] = df_eda["sbytes"] / (df_eda["dbytes"] + 1)
df_eda["log_byte_ratio"] = np.log1p(df_eda["byte_ratio"])
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="log_byte_ratio", data=df_eda)
plt.xlabel("Label (0 = Normal, 1 = Attack)")
plt.ylabel("Log(Byte Ratio)")
plt.title("Byte Asymmetry by Traffic Type")
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
sns.histplot(
    data=df_eda,
    x="log_byte_ratio",
    hue="Label",
    bins=100,
    element="step",
    stat="density",
    common_norm=False,
    alpha=0.6,
    palette={0: "blue", 1: "red"}
)
plt.xlabel("Log(Byte Ratio)")
plt.ylabel("Density")
plt.title("Distribution of Byte Asymmetry (Source / Destination Bytes)")
plt.show()

In [ ]:
# Scatter of Log(Byte Ratio) vs Log(Duration) by Label
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_eda,
    x="log_byte_ratio",
    y="log_dur",
    hue="Label",
    alpha=0.6,
    palette={0: "blue", 1: "red"}
)
plt.xlabel("Log(Byte Ratio)")
plt.ylabel("Log(Duration)")
plt.title("Byte Ratio vs Duration by Traffic Type")
plt.show()

In [ ]:
# Plot scaled distribution of "Sload" by label
df_eda["log_Sload"] = np.log1p(df_eda["Sload"])
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="log_Sload", data=df_eda)
plt.title("Distribution of Log(Source Load) by Label")
plt.show()

In [ ]:
# Plot scaled distribution of "Dload" by label
df_eda["log_Dload"] = np.log1p(df_eda["Dload"])
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="log_Dload", data=df_eda)
plt.title("Log(Destination Load) by Label")
plt.show()

In [ ]:
df_eda["log_load_skew"] = np.log1p(df_eda["Dload"]) - np.log1p(df_eda["Sload"])
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="log_load_skew", data=df_eda)
plt.title("Log(Load Skew) by Label")
plt.show()


In [ ]:
# # Inspect values
# proto_counts = df_eda["proto"].value_counts()
# display(proto_counts)

# # One-hot encode
# top_protocols = proto_counts.nlargest(6).index
# df_eda["proto_other"] = df_eda["proto"].apply(lambda x: 0 if x in top_protocols else 1)
# proto_dummies = pd.get_dummies(df_eda["proto"].where(df_eda["proto"].isin(top_protocols)), prefix="proto")
# df_eda = pd.concat([df_eda, proto_dummies], axis=1)

In [ ]:
df_eda["log_smeansz"] = np.log1p(df_eda["smeansz"])
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="log_smeansz", data=df_eda)
plt.title("Log(Source Mean Packet Size) by Label")
plt.show()

In [ ]:
df_eda["log_dmeansz"] = np.log1p(df_eda["dmeansz"])
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="log_dmeansz", data=df_eda)
plt.title("Log(Destination Mean Packet Size) by Label")
plt.show()

In [ ]:
df_eda["d_to_s_pkt_ratio"] = df_eda["dmeansz"]/(df_eda["smeansz"] + 1e-6)
df_eda["log_d_to_s_pkt_ratio"] = np.log1p(df_eda["d_to_s_pkt_ratio"])
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="log_d_to_s_pkt_ratio", data=df_eda)
plt.title("Distribution of Log(Dst/Source Avg Packet Size) by Label")
plt.show()

In [ ]:
# Plot distribution of "sttl" by label
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="sttl", data=df_eda)
plt.title("Distribution of Source Time to Live by Label")
plt.show()

In [ ]:
# Plot distribution of "dttl" by label
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="dttl", data=df_eda)
plt.title("Distribution of Destination Time to Live by Label")
plt.show()

plt.figure(figsize=(8, 6))
sns.histplot(df_eda[df_eda["Label"]==1]["dttl"], bins=30, kde=False)
plt.title("Histogram of Destination TTL for Attacks")
plt.xlabel("dttl")
plt.ylabel("Count")
plt.show()

In [ ]:
# Scatter relationship between "sttl" and "dttl"
plt.figure(figsize=(8, 6))
sns.scatterplot(x="sttl", y="dttl", hue="Label", data=df_eda, alpha=0.6)
plt.title("sttl vs dttl by Label")
plt.show()

# Comparative histograms
plt.figure(figsize=(8, 6))
sns.kdeplot(df_eda[df_eda['Label']==0]['dttl'], label='Normal', bw_adjust=0.5)
sns.kdeplot(df_eda[df_eda['Label']==1]['dttl'], label='Attack', bw_adjust=0.5)
plt.title("TTL Distribution by Label")
plt.legend()
plt.show()

# Highlight unusual TTL combinations
df_eda['ttl_diff'] = abs(df_eda['sttl'] - df_eda['dttl'])
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="ttl_diff", data=df_eda)
plt.title("Distribution of Time to Live Difference by Label")
plt.show()


In [ ]:
# Plot distribution of "Sjit" by label
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="Sjit", data=df_eda)
plt.title("Distribution of Source Jitter by Label")
plt.show()

In [ ]:
# Plot distribution of "Djit" by label
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="Djit", data=df_eda)
plt.title("Distribution of Destination Jitter by Label")
plt.show()

In [ ]:
# Experiment with temporal repetition
repetition_features = ["ct_srv_src", "ct_dst_ltm", "ct_src_dport_ltm", "ct_dst_sport_ltm", "ct_dst_src_ltm", "ct_src_ ltm", "ct_srv_dst"]
df_eda["repetition_strength"] = df_eda["ct_src_dport_ltm"]
df_eda["flow_volume_prod"] = np.log1p(df_eda["sbytes"]*df_eda["dbytes"])
df_eda["density"] = df_eda["flow_volume_prod"]/(1 + np.log1p(df_eda["dur"]))
df_eda["jitter_diff"] = abs(np.log1p(df_eda["Sjit"]) - np.log1p(df_eda["Djit"]))
df_eda["temporal_anomaly_score"] = (
    df_eda["repetition_strength"]/(1 + df_eda["jitter_diff"])
)
df_eda["log_temporal_anomaly_score"] = np.log1p(df_eda["temporal_anomaly_score"])
plt.figure(figsize=(8, 6))
sns.boxplot(x="Label", y="repetition_strength", data=df_eda)
plt.show()

In [ ]:
# Data leakage
display(df_eda["ct_state_ttl"].describe())

pd.crosstab(
    df_eda["ct_state_ttl"] > 0,
    df_eda["Label"],
    normalize="columns"
)

In [ ]:
# Categorical features
categorical_features = df_eda.select_dtypes(include=['object', 'category']).columns.tolist()
low_card_numeric = [col for col in df_eda.select_dtypes(include=[int, float]).columns if df_eda[col].nunique() <= 20]
categorical_features += low_card_numeric
print(categorical_features)

for col in ["sport", "dsport", "state", "service", "proto"]:
    print(f"\nValue counts for {col}:")
    print(df_eda[col].value_counts().head(10)) # top 10 categories

In [ ]:
attack_subset = df_eda[df_eda["Label"] == 1]

# Investigating state
attack_counts = attack_subset["state"].value_counts()
attack_frac = attack_counts/attack_counts.sum()
attack_state_df = pd.DataFrame({
    "attack_count": attack_counts,
    "attack_frac": attack_frac
}).sort_values(by="attack_count", ascending=False)
display(attack_state_df)

# Investigating service
attack_counts = attack_subset["service"].value_counts()
attack_frac = attack_counts/attack_counts.sum()
attack_state_df = pd.DataFrame({
    "attack_count": attack_counts,
    "attack_frac": attack_frac
}).sort_values(by="attack_count", ascending=False)
display(attack_state_df)

# Investigating proto
attack_counts = attack_subset["proto"].value_counts()
attack_frac = attack_counts/attack_counts.sum()
attack_state_df = pd.DataFrame({
    "attack_count": attack_counts,
    "attack_frac": attack_frac
}).sort_values(by="attack_count", ascending=False)
display(attack_state_df)

In [ ]:
normal_subset = df_eda[df_eda["Label"] == 0]

attack_counts = attack_subset["is_sm_ips_ports"].value_counts().sort_index()
normal_counts = normal_subset["is_sm_ips_ports"].value_counts().sort_index()
print(attack_counts, normal_counts)

df_counts = pd.DataFrame({
    "Normal": normal_counts,
    "Attack": attack_counts
}).fillna(0)

# Plot as bar chart
df_counts.plot(kind="bar", figsize=(8, 6))
plt.title("Distribution of is_sm_ips_ports by Label")
plt.xlabel("is_sm_ips_ports")
plt.ylabel("Number of flows")
plt.xticks([0, 1], ["0 (unique)", "1 (repeated)"], rotation=0)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(
    data=df_eda,
    x="trans_depth",
    hue="Label"
)
plt.title("Distribution of trans_depth by Label")
plt.xlabel("Transaction Depth")
plt.ylabel("Count")
plt.show()

display(
    df_eda[df_eda["Label"] == 0]["trans_depth"]
    .value_counts()
    .sort_index()
)

display(
    df_eda[df_eda["Label"] == 1]["trans_depth"]
    .value_counts()
    .sort_index()
)

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(
    data=df_eda,
    x="ct_flw_http_mthd",
    hue="Label"
)
plt.title("Distribution of ct_flw_http_mthd by Label")
plt.xlabel("Flow Method")
plt.ylabel("Count")
plt.show()

display(
    df_eda[df_eda["Label"] == 0]["ct_flw_http_mthd"]
    .value_counts()
    .sort_index()
)

display(
    df_eda[df_eda["Label"] == 1]["ct_flw_http_mthd"]
    .value_counts()
    .sort_index()
)


In [ ]:
# Save df_eda to CSV for modelling
df_eda.to_csv("../data/processed/df_eda.csv", index=False)